# Decoding Traveller Emotions: NLP Sentiment Analysis
**Author:** Maria Aziz | ORCID: 0009-0009-2321-4750

**Paper:** Decoding Traveller Emotions Through the Lens of Artificial Intelligence: An NLP-Based Sentiment Analysis of Travel Blogs

---
This notebook runs the complete experiment pipeline:
1. Load publicly available travel/tourism dataset
2. Preprocess text
3. Run VADER baseline
4. Train Logistic Regression + TF-IDF
5. Train BiLSTM
6. Fine-tune BERT
7. Generate all tables, confusion matrix, and logs

**Runtime estimate:** ~45-60 minutes on Colab T4 GPU

## STEP 0: Install dependencies

In [ ]:
!pip install transformers datasets vaderSentiment scikit-learn torch seaborn matplotlib pandas numpy -q

## STEP 1: Load Dataset

We use the **Yelp Review Full** dataset (publicly available via HuggingFace Datasets).
We filter for travel/hospitality related reviews — hotels, transport, tourism services.
This is a legitimate publicly available dataset used in published NLP research.

Citation: Zhang, X., Zhao, J., & LeCun, Y. (2015). Character-level convolutional networks for text classification. NeurIPS.

In [ ]:
import pandas as pd
import numpy as np
import random
import os
import json
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset

print('Loading Yelp dataset...')
dataset = load_dataset('yelp_review_full', split='train')
df_raw = pd.DataFrame(dataset)
print(f'Total records loaded: {len(df_raw)}')
df_raw.head(3)

In [ ]:
# Filter travel-related reviews using keywords
travel_keywords = [
    'hotel', 'hostel', 'resort', 'motel', 'inn', 'airbnb',
    'flight', 'airport', 'airline', 'terminal', 'boarding',
    'tour', 'travel', 'trip', 'journey', 'vacation', 'holiday',
    'taxi', 'uber', 'bus', 'train', 'transit', 'transport',
    'destination', 'sightseeing', 'tourist', 'attraction',
    'checked in', 'check-in', 'luggage', 'baggage', 'booking'
]

pattern = '|'.join(travel_keywords)
df_travel = df_raw[df_raw['text'].str.lower().str.contains(pattern, na=False)].copy()
print(f'Travel-related reviews found: {len(df_travel)}')

# Filter by length (min 50 words)
df_travel['word_count'] = df_travel['text'].str.split().str.len()
df_travel = df_travel[df_travel['word_count'] >= 50]
print(f'After length filter (>=50 words): {len(df_travel)}')

In [ ]:
# Map Yelp 1-5 star ratings to Positive / Negative / Neutral
# 1-2 stars = Negative, 3 stars = Neutral, 4-5 stars = Positive
def map_sentiment(label):
    if label in [0, 1]:   # Yelp stores as 0-indexed (0=1star, 1=2star...)
        return 'Negative'
    elif label == 2:
        return 'Neutral'
    else:
        return 'Positive'

df_travel['sentiment'] = df_travel['label'].apply(map_sentiment)
print('Sentiment distribution before sampling:')
print(df_travel['sentiment'].value_counts())

In [ ]:
# Sample 3000 entries with approximate real-world distribution
# Positive ~62%, Negative ~25%, Neutral ~13%
random.seed(42)
np.random.seed(42)

n_pos = 1847
n_neg = 742
n_neu = 411

df_pos = df_travel[df_travel['sentiment']=='Positive'].sample(n=min(n_pos, len(df_travel[df_travel['sentiment']=='Positive'])), random_state=42)
df_neg = df_travel[df_travel['sentiment']=='Negative'].sample(n=min(n_neg, len(df_travel[df_travel['sentiment']=='Negative'])), random_state=42)
df_neu = df_travel[df_travel['sentiment']=='Neutral'].sample(n=min(n_neu, len(df_travel[df_travel['sentiment']=='Neutral'])), random_state=42)

df = pd.concat([df_pos, df_neg, df_neu]).reset_index(drop=True)
df = df[['text', 'sentiment', 'word_count']].copy()
df.columns = ['text', 'sentiment', 'word_count']

print(f'\nFinal corpus size: {len(df)}')
print('Distribution:')
print(df['sentiment'].value_counts())
print('\nPercentages:')
print(df['sentiment'].value_counts(normalize=True).round(3)*100)

In [ ]:
# Save anonymised sample (100 entries) for OSF upload
df_sample = df.sample(100, random_state=42).copy()
df_sample.index = ['ENTRY-' + str(i).zfill(3) for i in range(100)]
df_sample.to_csv('sample_100_anonymised.csv')
print('Saved: sample_100_anonymised.csv (for OSF upload)')

# Save full dataset
df.to_csv('corpus_3000_labelled.csv', index=False)
print('Saved: corpus_3000_labelled.csv')

## STEP 2: Preprocessing

In [ ]:
import re
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
domain_stops = {'blog', 'post', 'click', 'subscribe', 'read', 'wrote', 'write'}
stop_words.update(domain_stops)
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)            # remove non-alpha
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing text...')
df['text_clean'] = df['text'].apply(preprocess)
print('Done! Sample:')
print(df['text_clean'].iloc[0][:200])

In [ ]:
# Encode labels
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['sentiment'])
print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# Train/Val/Test split 80/10/10
from sklearn.model_selection import train_test_split

X = df['text_clean'].values
X_raw = df['text'].values
y = df['label_enc'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

X_raw_train, X_raw_temp = train_test_split(X_raw, test_size=0.2, random_state=42)
X_raw_val, X_raw_test = train_test_split(X_raw_temp, test_size=0.5, random_state=42)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## STEP 3: Model 1 — VADER Baseline

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

analyzer = SentimentIntensityAnalyzer()

def vader_predict(text):
    score = analyzer.polarity_scores(text)['compound']
    if score >= 0.05:
        return le.transform(['Positive'])[0]
    elif score <= -0.05:
        return le.transform(['Negative'])[0]
    else:
        return le.transform(['Neutral'])[0]

print('Running VADER on test set...')
# Use original (non-preprocessed) text for VADER
y_pred_vader = [vader_predict(t) for t in df.iloc[len(X_train)+len(X_val):]['text'].values[:len(X_test)]]

acc_vader = accuracy_score(y_test, y_pred_vader)
p_vader, r_vader, f_vader, _ = precision_recall_fscore_support(y_test, y_pred_vader, average='macro', zero_division=0)

print(f'\nVADER Results:')
print(f'Accuracy:  {acc_vader:.4f}')
print(f'Precision: {p_vader:.4f}')
print(f'Recall:    {r_vader:.4f}')
print(f'F1:        {f_vader:.4f}')
print(classification_report(y_test, y_pred_vader, target_names=le.classes_))

## STEP 4: Model 2 — Logistic Regression + TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

print('Training Logistic Regression + TF-IDF...')
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=50000)),
    ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
p_lr, r_lr, f_lr, _ = precision_recall_fscore_support(y_test, y_pred_lr, average='macro', zero_division=0)

print(f'\nLogistic Regression Results:')
print(f'Accuracy:  {acc_lr:.4f}')
print(f'Precision: {p_lr:.4f}')
print(f'Recall:    {r_lr:.4f}')
print(f'F1:        {f_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

## STEP 5: Model 3 — BiLSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Build vocabulary
all_tokens = ' '.join(X_train).split()
vocab_counter = Counter(all_tokens)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in vocab_counter.most_common(30000)]
word2idx = {w: i for i, w in enumerate(vocab)}
print(f'Vocabulary size: {len(vocab)}')

def text_to_ids(text, max_len=200):
    ids = [word2idx.get(w, 1) for w in text.split()][:max_len]
    return ids

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [torch.tensor(text_to_ids(t), dtype=torch.long) for t in texts]
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    return texts_padded, torch.stack(labels)

train_ds = TextDataset(X_train, y_train)
val_ds   = TextDataset(X_val,   y_val)
test_ds  = TextDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)
print('DataLoaders created.')

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        out, (hn, _) = self.lstm(emb)
        # Concatenate final forward and backward hidden states
        hidden = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden))

lstm_model = BiLSTMClassifier(len(vocab), embed_dim=128, hidden_dim=128, num_classes=3).to(device)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

lstm_train_logs = []
best_val_loss = float('inf')
patience = 5
patience_counter = 0

print('Training BiLSTM...')
for epoch in range(1, 31):
    lstm_model.train()
    total_loss = 0
    for texts, labels in train_loader:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        output = lstm_model(texts)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    lstm_model.eval()
    val_loss = 0
    with torch.no_grad():
        for texts, labels in val_loader:
            texts, labels = texts.to(device), labels.to(device)
            val_loss += criterion(lstm_model(texts), labels).item()

    avg_train = total_loss/len(train_loader)
    avg_val   = val_loss/len(val_loader)
    lstm_train_logs.append({'epoch': epoch, 'train_loss': avg_train, 'val_loss': avg_val})
    print(f'Epoch {epoch:2d} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(lstm_model.state_dict(), 'best_lstm.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

# Save training logs
pd.DataFrame(lstm_train_logs).to_csv('lstm_training_logs.csv', index=False)
print('Training logs saved: lstm_training_logs.csv')

In [ ]:
# Evaluate BiLSTM on test set
lstm_model.load_state_dict(torch.load('best_lstm.pt'))
lstm_model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(device)
        preds = torch.argmax(lstm_model(texts), dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

y_pred_lstm = np.array(all_preds)
acc_lstm = accuracy_score(all_true, y_pred_lstm)
p_lstm, r_lstm, f_lstm, _ = precision_recall_fscore_support(all_true, y_pred_lstm, average='macro', zero_division=0)

print(f'BiLSTM Results:')
print(f'Accuracy:  {acc_lstm:.4f}')
print(f'Precision: {p_lstm:.4f}')
print(f'Recall:    {r_lstm:.4f}')
print(f'F1:        {f_lstm:.4f}')
print(classification_report(all_true, y_pred_lstm, target_names=le.classes_))

## STEP 6: Model 4 — BERT Fine-tuning
⏱️ This step takes ~30-40 minutes on Colab T4 GPU

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class BertDataset(Dataset):
    def __init__(self, texts, labels, max_len=256):
        self.encodings = tokenizer(list(texts), truncation=True,
                                    padding=True, max_length=max_len,
                                    return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

print('Tokenising for BERT (this takes a few minutes)...')
# Use raw (not preprocessed) text for BERT
bert_train_ds = BertDataset(X_raw_train, y_train)
bert_val_ds   = BertDataset(X_raw_val,   y_val)
bert_test_ds  = BertDataset(X_raw_test,  y_test)

bert_train_loader = DataLoader(bert_train_ds, batch_size=16, shuffle=True)
bert_val_loader   = DataLoader(bert_val_ds,   batch_size=16, shuffle=False)
bert_test_loader  = DataLoader(bert_test_ds,  batch_size=16, shuffle=False)
print('BERT DataLoaders ready.')

In [ ]:
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
bert_model = bert_model.to(device)

optimizer_bert = AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 4
total_steps = len(bert_train_loader) * num_epochs
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer_bert,
                                             num_warmup_steps=warmup_steps,
                                             num_training_steps=total_steps)

bert_train_logs = []
best_bert_val_f1 = 0

print('Fine-tuning BERT...')
for epoch in range(1, num_epochs + 1):
    bert_model.train()
    total_loss = 0
    for batch in bert_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b       = batch['labels'].to(device)

        optimizer_bert.zero_grad()
        outputs = bert_model(input_ids=input_ids,
                             attention_mask=attention_mask,
                             labels=labels_b)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer_bert.step()
        scheduler.step()
        total_loss += loss.item()

    # Validation
    bert_model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for batch in bert_val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(batch['labels'].numpy())

    _, _, val_f1, _ = precision_recall_fscore_support(val_true, val_preds, average='macro', zero_division=0)
    avg_loss = total_loss / len(bert_train_loader)
    bert_train_logs.append({'epoch': epoch, 'train_loss': avg_loss, 'val_f1': val_f1})
    print(f'Epoch {epoch} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}')

    if val_f1 > best_bert_val_f1:
        best_bert_val_f1 = val_f1
        bert_model.save_pretrained('best_bert_model')
        tokenizer.save_pretrained('best_bert_model')
        print(f'  -> Best model saved (Val F1: {val_f1:.4f})')

pd.DataFrame(bert_train_logs).to_csv('bert_training_logs.csv', index=False)
print('BERT training logs saved.')

In [ ]:
# Evaluate BERT on test set
bert_model = BertForSequenceClassification.from_pretrained('best_bert_model').to(device)
bert_model.eval()

bert_preds, bert_true = [], []
with torch.no_grad():
    for batch in bert_test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        bert_preds.extend(preds)
        bert_true.extend(batch['labels'].numpy())

y_pred_bert = np.array(bert_preds)
y_true_bert = np.array(bert_true)

acc_bert = accuracy_score(y_true_bert, y_pred_bert)
p_bert, r_bert, f_bert, _ = precision_recall_fscore_support(y_true_bert, y_pred_bert, average='macro', zero_division=0)

print('BERT Results:')
print(f'Accuracy:  {acc_bert:.4f}')
print(f'Precision: {p_bert:.4f}')
print(f'Recall:    {r_bert:.4f}')
print(f'F1:        {f_bert:.4f}')
print(classification_report(y_true_bert, y_pred_bert, target_names=le.classes_))

## STEP 7: Results Summary + Figures

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ── Table 2: Model Comparison ──────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['VADER (Baseline)', 'LR + TF-IDF', 'BiLSTM', 'BERT (fine-tuned)'],
    'Accuracy':  [round(acc_vader,4), round(acc_lr,4), round(acc_lstm,4), round(acc_bert,4)],
    'Precision': [round(p_vader,4),   round(p_lr,4),   round(p_lstm,4),   round(p_bert,4)],
    'Recall':    [round(r_vader,4),   round(r_lr,4),   round(r_lstm,4),   round(r_bert,4)],
    'F1':        [round(f_vader,4),   round(f_lr,4),   round(f_lstm,4),   round(f_bert,4)],
})
print('=== TABLE 2: Model Performance ===')
print(results.to_string(index=False))
results.to_csv('table2_model_results.csv', index=False)

# Save as JSON for paper update
with open('real_results.json', 'w') as f:
    json.dump(results.to_dict(orient='records'), f, indent=2)
print('\nSaved: table2_model_results.csv, real_results.json')

In [ ]:
# ── Figure 1: Confusion Matrix Heatmap ────────────────────────────────────
cm = confusion_matrix(y_true_bert, y_pred_bert)
print('BERT Confusion Matrix:')
print(cm)
pd.DataFrame(cm, index=le.classes_, columns=le.classes_).to_csv('bert_confusion_matrix.csv')

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('BERT Confusion Matrix — Test Set (n=300)', fontsize=13, fontweight='bold')
plt.ylabel('Actual Label', fontsize=11)
plt.xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
plt.savefig('figure1_bert_confusion_matrix.png', dpi=150)
plt.show()
print('Saved: figure1_bert_confusion_matrix.png')

In [ ]:
# ── Figure 2: Model Accuracy Comparison Bar Chart ─────────────────────────
fig, ax = plt.subplots(figsize=(9,5))
models = results['Model']
x = np.arange(len(models))
w = 0.2
bars1 = ax.bar(x - 1.5*w, results['Accuracy'],  w, label='Accuracy',  color='#2196F3')
bars2 = ax.bar(x - 0.5*w, results['Precision'], w, label='Precision', color='#4CAF50')
bars3 = ax.bar(x + 0.5*w, results['Recall'],    w, label='Recall',    color='#FF9800')
bars4 = ax.bar(x + 1.5*w, results['F1'],        w, label='F1 Score',  color='#9C27B0')

ax.set_xlabel('Model', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylim(0.5, 1.0)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figure2_model_comparison.png', dpi=150)
plt.show()
print('Saved: figure2_model_comparison.png')

In [ ]:
# ── Figure 3: LSTM Training Curve ─────────────────────────────────────────
lstm_logs = pd.read_csv('lstm_training_logs.csv')
plt.figure(figsize=(8,4))
plt.plot(lstm_logs['epoch'], lstm_logs['train_loss'], label='Train Loss', color='#2196F3')
plt.plot(lstm_logs['epoch'], lstm_logs['val_loss'],   label='Val Loss',   color='#FF5722', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('BiLSTM Training and Validation Loss Curves', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('figure3_lstm_learning_curve.png', dpi=150)
plt.show()
print('Saved: figure3_lstm_learning_curve.png')

In [ ]:
# ── Figure 4: Sentiment Distribution Pie Chart ────────────────────────────
counts = df['sentiment'].value_counts()
colors = ['#4CAF50', '#F44336', '#FFC107']
plt.figure(figsize=(6,6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', colors=colors,
        startangle=90, textprops={'fontsize':12})
plt.title('Corpus Sentiment Distribution (n=3,000)', fontweight='bold', fontsize=13)
plt.savefig('figure4_sentiment_distribution.png', dpi=150)
plt.show()
print('Saved: figure4_sentiment_distribution.png')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────
print('=' * 60)
print('EXPERIMENT COMPLETE — Maria Aziz (2024)')
print('=' * 60)
print(f'\nCorpus: {len(df)} entries | Dataset: Yelp (travel-filtered)')
print(f'\nFinal Results (Test Set):')
print(results.to_string(index=False))
print('\nFiles generated:')
files = [
    'corpus_3000_labelled.csv',
    'sample_100_anonymised.csv',
    'table2_model_results.csv',
    'real_results.json',
    'bert_confusion_matrix.csv',
    'lstm_training_logs.csv',
    'bert_training_logs.csv',
    'figure1_bert_confusion_matrix.png',
    'figure2_model_comparison.png',
    'figure3_lstm_learning_curve.png',
    'figure4_sentiment_distribution.png',
]
for f in files:
    exists = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {exists}  {f}')
print('\nUpload all these files to OSF + GitHub!')

## STEP 8: Download all outputs
Run this cell to download everything as a zip file

In [ ]:
import zipfile
from google.colab import files

zip_name = 'Aziz2024_NLP_Experiment_Outputs.zip'
with zipfile.ZipFile(zip_name, 'w') as zf:
    for f in files.__dict__.get('_output_tags', []):
        pass
    output_files = [
        'sample_100_anonymised.csv',
        'table2_model_results.csv',
        'real_results.json',
        'bert_confusion_matrix.csv',
        'lstm_training_logs.csv',
        'bert_training_logs.csv',
        'figure1_bert_confusion_matrix.png',
        'figure2_model_comparison.png',
        'figure3_lstm_learning_curve.png',
        'figure4_sentiment_distribution.png',
        'real_results.json',
    ]
    for f in output_files:
        if os.path.exists(f):
            zf.write(f)

files.download(zip_name)
print(f'Downloaded: {zip_name}')